# Binaryzacja


### Cel:
- zapoznanie z segmentacją obiektów poprzez binaryzację,
- zapoznanie z binaryzacją na podstawie histogramu (globalną),
- zapoznanie z metodami automatycznego wyznaczania progu Otsu, Kitllera i Kapura,
- zapoznanie z binaryzacją lokalną (na podstawie średniej i metodą Sauvoli),
- zapoznanie z binaryzacją dwuprogową,
- zadanie domowe: zapoznanie z adaptacyjną binaryzacją lokalną.

### Binaryzacja - wprowadzenie

Jednym z najważniejszych etapów podczas analizy obrazów jest segmentacja -- podział obrazu na rejony według pewnego kryterium  -- jasności, koloru, tekstury.
Najprostszą (i też najczęściej wykorzystywaną) metodą segmentacji jest **binaryzacja**. 
Do jej głównych zalet zalicza się: intuicyjność, prostotę, łatwość implementacji i szybkość wykonywania. 
Jest ona etapem wielu algorytmów analizy obrazów. 
Pozwala na znaczną redukcję informacji w obrazie (np. dla wejściowego obrazu w skali szarości z zakresu 0-255 do 0-1).
 
Binaryzacja najczęściej realizowana jest poprzez progowanie. 
Na przykład: dla obrazu w odcieniach szarości ustala się próg na poziomie $k$. 
Wszystkie piksele o wartości (jasności) większej od $k$ zostają uznane za obiekty, a pozostałe za tło. 
Oczywiście podejście takie daje się zastosować wtedy, gdy obiekty mają istotnie różną jasność od otaczającego je tła.


### Binaryzacja na podstawie histogramu

W rozdziale zostanie zademonstrowane wyznaczanie progu na podstawie "ręcznej" analizy histogramu oraz wpływ szumu i niejednorodnego oświetlenia sceny na proces binaryzacji.

1. Potrzebne w ćwiczeniu moduły są już wpisane - zwróć uwagę pod jakimi nazwami będą one widziane (plt, cv2, np).

2. Wczytaj obraz _coins.png_ w trybie odcieni szarości. Wyświetl go. 
Wyznacz jego histogram (funkcja `np.histogram` lub 'cv2.calcHist') i wyświetl go.
Przy wyświetlaniu histogramu warto zwiększyć liczbę wyświetlanych wartości na osi x oraz powiększyć sam wykres (funkcje *plt.xticks(np.arange(0, 256, 20.0))* oraz *plt.rcParams["figure.figsize"] = (10,5)*.
Uwaga. Proszę powyższą funkcjonalność zaimplementować jako funkcję, gdyż przyda się w dalszej części ćwiczenia.
      


In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os

if not os.path.exists("coins.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/coins.png --no-check-certificate
if not os.path.exists("rice.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/rice.png --no-check-certificate
if not os.path.exists("catalogue.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/catalogue.png --no-check-certificate
if not os.path.exists("bart.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/bart.png --no-check-certificate
if not os.path.exists("figura1.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/figura1.png --no-check-certificate
if not os.path.exists("figura2.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/figura2.png --no-check-certificate
if not os.path.exists("figura3.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/figura3.png --no-check-certificate
if not os.path.exists("figura4.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/figura4.png --no-check-certificate

def load_and_display_with_histogram(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    
    plt.figure()
    plt.imshow(image, cmap='gray')
    plt.title("Obraz - Coins.png")
    plt.show()
    
    hist = cv2.calcHist([image], [0], None, [256], [0, 256])
    
    plt.figure(figsize=(10, 5))
    plt.plot(hist)
    plt.xlim([0, 256])
    plt.xticks(np.arange(0, 256, 20.0))
    plt.title("Histogram")
    plt.grid(True)
    plt.show()
    
    return image, hist

image, hist = load_and_display_with_histogram("coins.png")

3. Wizualna analiza histogramu pozwala zauważyć dwa maksima - jedno odpowiadające poziomowi jasności tła (które w tym przypadku jest względnie jednolite - ciemnoszare) i drugie odpowiadające monetom.

Na podstawie histogramu wyznacz próg i wykonaj binaryzację:
- wykorzystaj fakt, że dla macierzy *numpy* można wykonać operację porównania wszystkich jej wartości z liczbą  - wynikiem jest macierz zawierająca wartości *True* i *False*, którą można przekonwertować metodą macierz.astype(np.int) na macierz z wartościami 1 i 0 (aczkolwiek nie jest to tu konieczne).
- wynik binaryzacji wyświetl,
- spróbuj dobrać jak najlepszy próg binaryzacji. Jako "kryterium jakości" przyjmij kształty monet - dla poprawnie dobranego progu powinny to być wypełnione koła.

Uwaga. Proszę powyższą funkcjonalność zaimplementować jako funkcję, gdyż przyda się w dalszej części ćwiczenia.

In [ ]:
def binary_threshold(image, threshold):
    """
    Binaryzacja obrazu na podstawie podanego progu.
    
    Args:
        image: Obraz wejściowy w skali szarości
        threshold: Próg binaryzacji (0-255)
        
    Returns:
        Zbinaryzowany obraz
    """
    binary_image = (image > threshold).astype(np.uint8) * 255
    
    plt.figure(figsize=(10, 5))
    plt.subplot(121)
    plt.imshow(image, cmap='gray')
    plt.title(f"Obraz oryginalny")
    
    plt.subplot(122)
    plt.imshow(binary_image, cmap='gray')
    plt.title(f"Binaryzacja (próg={threshold})")
    plt.show()
    
    return binary_image

binary_image_120 = binary_threshold(image, 120)
binary_image_140 = binary_threshold(image, 140)
binary_image_160 = binary_threshold(image, 160)

# Prezentacja najlepszego wyniku binaryzacji
print("="*50)
print("NAJLEPSZY WYNIK BINARYZACJI - PRÓG 140")
print("="*50)
#! Po obejrzeniu wyników powyżej, wybierzmy najlepszy próg to 140
best_threshold = 140 
best_binary_image = binary_threshold(image, best_threshold)

4. Na "stopień trudności" przeprowadzenia binaryzacji największy wpływ mają dwa czynniki:
- szum,
- niejednorodne oświetlenie.
	  
Użyj obrazy:
 - _figura1.png_ (bez zaszumienia),
 - _figura2.png_ (dodany szum Gaussowski o średniej 0 i odchyleniu standardowym 10),
 - _figura3.png_ (dodany szum Gaussowski o średniej 0 i odchyleniu standardowym 50),
 - _figura4.png_ (dodany gradient oświetlenia -- symulacja oświetlenia niejednorodnego) i wyświetl ich histogramy (wykorzystaj funkcję z poprzedniego punktu).


In [ ]:
# Wczytanie i wyświetlenie obrazów wraz z ich histogramami
print("OBRAZ FIGURA1 - BEZ ZASZUMIENIA")
print("="*50)
figura1, hist1 = load_and_display_with_histogram("figura1.png")

print("\nOBRAZ FIGURA2 - SZUM GAUSSOWSKI (śr=0, odch.std=10)")
print("="*50)
figura2, hist2 = load_and_display_with_histogram("figura2.png")

print("\nOBRAZ FIGURA3 - SZUM GAUSSOWSKI (śr=0, odch.std=50)")
print("="*50)
figura3, hist3 = load_and_display_with_histogram("figura3.png")

print("\nOBRAZ FIGURA4 - GRADIENT OŚWIETLENIA")
print("="*50)
figura4, hist4 = load_and_display_with_histogram("figura4.png")

Spróbuj wyznaczyć progi binaryzacji na podstawie wyświetlonych histogramów.
Jak dodanie szumu wypłynęło na histogram i łatwość wyznaczania progu binaryzacji?
Czy jest to możliwe we wszystkich przypadkach?

In [ ]:
def binary_threshold_all_images():
    threshold1 = 100
    threshold2 = 128  
    threshold3 = 128  
    threshold4 = 128  
    
    print("BINARYZACJA OBRAZU BEZ SZUMU (figura1.png)")
    binary1 = binary_threshold(figura1, threshold1)
    
    print("\nBINARYZACJA OBRAZU Z MAŁYM SZUMEM (figura2.png)")
    binary2 = binary_threshold(figura2, threshold2)
    
    print("\nBINARYZACJA OBRAZU Z DUŻYM SZUMEM (figura3.png)")
    binary3 = binary_threshold(figura3, threshold3)
    
    print("\nBINARYZACJA OBRAZU Z GRADIENTEM OŚWIETLENIA (figura4.png)")
    binary4 = binary_threshold(figura4, threshold4)
    
    return binary1, binary2, binary3, binary4

binary1, binary2, binary3, binary4 = binary_threshold_all_images()

# Analiza binaryzacji obrazów z różnym poziomem szumu

### Obraz bez szumu (figura1.png)

*   Histogram prawdopodobnie prezentuje wyraźny rozkład bimodalny - dwa dobrze oddzielone szczyty
*   Próg binaryzacji można łatwo wyznaczyć jako dolinę między tymi szczytami
*   Wartość 100 wydaje się odpowiednia, co wskazuje na wyraźne rozdzielenie obiektów od tła

### Obraz z małym szumem (figura2.png)

*   Szum powoduje częściowe "rozmycie" szczytów histogramu
*   Dolina między pikselami obiektu i tła staje się mniej wyraźna
*   Wartość 128 jest rozsądnym kompromisem, ale może wymagać dostrojenia

### Obraz z dużym szumem (figura3.png)

*   Histogram jest znacząco zaburzony przez szum
*   Szczyt i dolina są trudniejsze do zidentyfikowania wizualnie
*   Wartość 128 może nie być optymalna - warto zastosować algorytm iteracyjny (jak `iter_binary`)
*   Przy dużym szumie, prosta binaryzacja może dawać niezadowalające wyniki

### Obraz z gradientem oświetlenia (figura4.png)

*   Gradient powoduje, że ten sam obiekt ma różną jasność w różnych obszarach obrazu
*   Histogram może nie być bimodalny, a raczej rozmyty
*   Pojedynczy globalny próg (128) może nie działać poprawnie w całym obrazie

## Wpływ szumu na histogram i proces binaryzacji

Szum ma istotny wpływ na proces binaryzacji:

*   Zaburzenie bimodalności histogramu - szum powoduje rozmycie szczytów i wypełnienie doliny
*   Zwiększona trudność wyznaczania progu - mniej wyraźna granica między klasami pikseli
*   Niepewność klasyfikacji - piksele o wartościach bliskich progowi mogą zostać niepoprawnie sklasyfikowane
*   Potrzeba zaawansowanych metod - przy większym szumie proste metody progowania stają się niewystarczające

## Czy binaryzacja jest zawsze możliwa?

Nie we wszystkich przypadkach możliwe jest poprawne wyznaczenie pojedynczego progu binaryzacji:

*   Obrazy z gradientem oświetlenia - wymagają binaryzacji adaptacyjnej, która stosuje różne progi w różnych obszarach obrazu
*   Obrazy z dużym zaszumieniem - mogą wymagać wstępnej filtracji (np. filtr medianowy)
*   Obrazy wielomodalne - gdy histogram ma więcej niż dwa szczyty (wiele klas obiektów)
*   Obrazy o niskim kontraście - gdy różnica między obiektem a tłem jest minimalna

## Rekomendacje

*   Dla obrazów z szumem: przed binaryzacją warto zastosować filtrację
*   Dla obrazów z nierównomiernym oświetleniem: zastosować binaryzację adaptacyjną (np. Otsu lokalnie)
*   Dla trudnych przypadków: rozważyć algorytmy `iter_binary` (jak pokazano w kodzie) lub inne zaawansowane metody segmentacji

Funkcja `iter_binary` w kodzie implementuje iteracyjną metodę znajdowania progu, która jest bardziej odporna na szum niż ręczne ustalanie progów i mogłaby poprawić wyniki binaryzacji dla problematycznych obrazów.
```

### Automatyczne wyznaczanie progu binaryzacji

W automatycznym systemie analizy obrazów (działanie bez nadzoru operatora) konieczne jest zastosowanie metody binaryzacji, która w sposób automatyczny wyznacza próg binaryzacji.
Oczywiście można sobie wyobrazić użycie stałego progu (np. 10), ale wtedy należy zadbać o niezmienność warunków oświetleniowych, co w niektórych zastosowaniach może być problematyczne.

#### Iteracyjne wyznaczenie progu

Jednym z najprostszych podejść jest iteracyjna procedura wyliczania progu.
Jako pierwsze przybliżenie progu ($k$) przyjmuje się średnia jasność na obrazie.
Następnie, na podstawie $k$,  dzieli się obraz na dwa podobrazy $I_0$ i  $I_1$ (dwie klasy $C_0$ i $C_1$).
Dla każdego z nich oblicza się średnią jasność: $m_0$ i $m_1$.
Jako nowy próg przyjmuje się:

\begin{equation}
k_{new} = \frac{m_0 + m_1}{2}
\tag{1}
\end{equation}

Procedurę kontynuuje się do momentu, aż różnica pomiędzy dwoma kolejnymi progami będzie mniejsza niż zadana wartość.


**Zadanie: zaimplementować opisany powyżej algorytm.**


Jak można zauważyć, do poprawnego działania metody potrzebne będzie obliczanie średniej jasności, również dla pewnych podobrazów.
Wykorzystamy do tego znormalizowany histogram:
\begin{equation}
\tag{2}
p_i = n_i/N,   \sum_{i=0}^L p_i = 1
\end{equation}
gdzie: $n_i$ liczba pikseli o jasności $i$ ($i = 0,1, ... L-1$) - histogram, $L$ - liczba poziomów jasności, $N$ - liczba pikseli na obrazie ($N = n_0 + n_1 + ... + n_{L-1}$).

Jeśli podzielimy obraz na dwie klasy $C_0$ i $C_1$ (tło i obiekty albo obiekty i tło) z progiem podziału oznaczonym jako $k$, to do klasy $C_0$ należeć będą piksele o poziomach $[0,k]$, a do klasy $C1$ piksele o poziomach $[k+1,L-1]$.

Wtedy prawdopodobieństwo, że piksel należy do klasy $C_0$ wynosi:
\begin{equation}
\tag{3}
P_0(k) = \sum_{i=0}^{k} p_i
\end{equation}

Podobnie prawdopodobieństwo, że należy do klasy $C_1$ wynosi:

\begin{equation}
\tag{4}
P_1(k) = \sum_{i=k+1}^{L-1} p_i = 1 - P_0(k)
\end{equation}

Średnią jasność pikseli należących do klasy $C_0$ można wyznaczyć na podstawie:

\begin{equation}
\tag{5}
m_0(k) = \sum_{i=0}^{k} iP(i|C_0)
\end{equation}

gdzie: $|$ oznacza prawdopodobieństwo warunkowe, a wyraz $P(i|C_0)$ - prawdopodobieństwo dla wartości $i$ pod warunkiem, że $i$ należy do klasy $C_0$.
Równanie to jest szczególnym przypadkiem wykorzystania momentów statystycznych do wyliczania pewnych parametrów statystycznych - w tym przypadku średniej.

Wykorzystując regułę Bayesa:

\begin{equation}
\tag{6}
P(A|B) = P(B|A)P(A)/P(B)
\end{equation}
możemy zapisać:

\begin{equation}
\tag{7}
m_0(k) = \sum_{i=0}^{k} i P(C_0|i)P(i)/P(C_0)
\end{equation}
Wyraz $P(C_0|i) = 1$, gdyż z założenia rozpatrujemy tylko piksele należące do klasy $C_0$.
Wyraz $P(i)$ stanowi $i$-ty element znormalizowanego histogramu tj. $P(i) = p_i$, a $P(C_0)$ to prawdopodobieństwo przynależności do klasy $C_0$ określone wcześniej $P(C_0) = P_0(k)$.
Ostatecznie możemy więc zapisać:

\begin{equation}
\tag{8}
m_0(k) = \frac{1}{P_0(k)} \sum_{i=0}^{k} i p_i
\end{equation}

Na podstawie analogicznych rozważań można wyprowadzić wzór na średnią jasności pikseli należących do klasy $C_1$:
\begin{equation}
\tag{9}
m_1(k) = \frac{1}{P_1(k)} \sum_{i=k+1}^{L-1} i p_i
\end{equation}

Średnia jasność całego obrazu dana jest zależnością:
\begin{equation}
\tag{10}
m_G = \sum_{i=0}^{L-1} ip_i
\end{equation}


1. Wczytaj obraz _coins.png_. Wyświetl go.

2. Wylicz histogram i histogram skumulowany (funkcja `np.cumsum`).
   Na podstawie zależności $(10)$ wylicz średnią - pierwszy próg podziału $k$.
   Uwagi:
   - przed dalszymi obliczeniami dobrze jest usunąć zbędny wymiar tablicy z histogramem - polecenie `np.squeeze`
    - $p_i$ to nasz znormalizowany histogram, a wartości od $0$ do $255$ można wygenerować poleceniem `np.arange(256)`,
    - zmiast pętli `for` można wykorzystać iloczyn sklarny dwóch wektorów tj. `np.dot`.

3.  W nieskończonej petli `while` wykonaj następujące kroki:
- oblicz średnią $m_0$ -- zależność $(8)$:
    - dla $P_0$ wystarczy wykorzystać odpowiednią wartość znormalizowanego histogramu skumulowanego, dla pozostałej części wyrażenia podobne rozwiązanie jak dla pierwszej średniej,
- oblicz średnią $m_1$ -- zależność $(9)$,
- oblicz nowy próg $k_{new}$ -- zależność $(1)$,
- oblicz moduł z różnicy pomiędzy $k_{new}$, a $k$ i sprawdź czy jest mniejszy od progu (np. $1$),
- jeśli tak to zakończ obliczenia (`break`), jeśli nie to przypisz $k = k_{new}$ i kontynuuj obliczenia,
- wyświetl próg oraz wynik binaryzacji.

4. Sprawdź jak metoda działa na obrazach _figura1.png_ do _figura4.png_. 

In [ ]:
import cv2
import numpy as np

def iterative_threshold(image):
    hist = cv2.calcHist([image], [0], None, [256], [0, 256])
    hist = np.squeeze(hist)
    pixel_count = image.shape[0] * image.shape[1]
    normalized_hist = hist / pixel_count
    cumulative_hist = np.cumsum(normalized_hist)
    intensity_levels = np.arange(256)
    mG = np.dot(intensity_levels, normalized_hist)
    k = int(mG)
    threshold = 1.0
    iteration = 0
    
    while True:
        iteration += 1
        P0 = cumulative_hist[k]
        sum_ipi_C0 = np.dot(intensity_levels[:k+1], normalized_hist[:k+1])
        m0 = sum_ipi_C0 / P0 if P0 > 0 else 0
        P1 = 1 - P0
        sum_ipi_C1 = np.dot(intensity_levels[k+1:], normalized_hist[k+1:])
        m1 = sum_ipi_C1 / P1 if P1 > 0 else 0
        k_new = int((m0 + m1) / 2)
        
        if abs(k_new - k) < threshold:
            break
        
        k = k_new
    
    binary_image = binary_threshold(image, k)
    
    return k, binary_image

print("ITERACYJNE WYZNACZANIE PROGU DLA OBRAZU COINS.PNG")
print("="*50)
threshold_coins, binary_coins = iterative_threshold(image)

print("\nITERACYJNE WYZNACZANIE PROGU DLA OBRAZU FIGURA1.PNG (bez szumu)")
print("="*50)
threshold_fig1, binary_fig1 = iterative_threshold(figura1)

print("\nITERACYJNE WYZNACZANIE PROGU DLA OBRAZU FIGURA2.PNG (mały szum)")
print("="*50)
threshold_fig2, binary_fig2 = iterative_threshold(figura2)

print("\nITERACYJNE WYZNACZANIE PROGU DLA OBRAZU FIGURA3.PNG (duży szum)")
print("="*50)
threshold_fig3, binary_fig3 = iterative_threshold(figura3)

print("\nITERACYJNE WYZNACZANIE PROGU DLA OBRAZU FIGURA4.PNG (gradient)")
print("="*50)
threshold_fig4, binary_fig4 = iterative_threshold(figura4)

print("\nPORÓWNANIE WYZNACZONYCH PROGÓW:")
print(f"Obraz coins.png: {threshold_coins}")
print(f"Obraz figura1.png: {threshold_fig1}")
print(f"Obraz figura2.png: {threshold_fig2}")
print(f"Obraz figura3.png: {threshold_fig3}")
print(f"Obraz figura4.png: {threshold_fig4}")

#### Metoda Otsu

Jednym z częściej wykorzystywanych algorytmów wyznaczania progu jest metoda zaproponowana w roku 1979 przez Nobuyuki Otsu w artykule pt. "A Threshold Selection Method from Gray-Level Histograms" (można odszukać na IEEE Xplore).
W algorytmie zakłada się, że obraz zawiera piksele należące do dwóch klas (obiektów i tła) tj. histogram obrazu jest bi-modalny (ma dwa maksima).
Próg podziału obliczany jest tak, aby wariancja międzyklasowa była maksymalna.
W tym sensie metodę Otsu można nazwać optymalną.

Wprowadźmy teraz wskaźnik "jakości" wybranego progu podziału $k$, który będziemy optymalizować.
W algorytmie Otsu jest to:

\begin{equation}
\tag{11}
\eta(k) = \frac{\sigma^2_B(k)}{\sigma^2_G}
\end{equation}
gdzie:  $\sigma^2_G$ - wariancja globalna, która może zostać obliczona na podstawie momentów statystycznych jako:

\begin{equation}
\tag{12}
\sigma^2_G =  \sum_{i=0}^{L-1} (i - m_G)^2 p_i
\end{equation}
a $\sigma^2_B$ jest wariancją międzyklasową, która jest zdefiniowana jako:
\begin{equation}
\tag{13}
\sigma^2_B(k) =  P_0(k)(m_0(k) - m_G)^2 + P_1(k)(m_1(k) - m_G)^2
\end{equation}
Równianie to można również przekształcić do:
\begin{equation}
\tag{14}
\sigma^2_B(k) =  P_0(k)P_1(k)(m_0(k) - m_1(k))^2 = \frac{(m_G P_0(k) - m(k) )^2}{P_0(k)(1-P_0(k))}
\end{equation}
gdzie:
\begin{equation}
\tag{15}
m(k) = \sum_{i=0}^{k} i p_i
\end{equation}

Taki zapis pozwala przyspieszyć obliczenia.
Wartość $m_G$ wyznaczana jest jednokrotnie, a zachodzi tylko potrzeba obliczania $m(k)$ i $P_0(k)$ w każdej iteracji.
Warto też zwrócić uwagę, że równanie ma sens dla $P_0 > 0$.

Warto zauważyć, że z postaci równania $(14)$ wynika, że im większa odległość pomiędzy średnimi $m_0$ i $m_1$ tym wartość wariancji międzyklasowej jest większa.
Pokazuje to, że przyjęty współczynniki może być podstawą do separacji dwóch klas - im jego wartość jest większa, tym lepsze rozdzielenie.
Dodatkowo, z równania $(11)$ wynika, że $\eta(k)$ zależy tylko od wariancji międzyklasowej $\sigma^2_B(k)$, gdyż wariancja globalna $\sigma^2_G$ jest stała.
Zatem w procesie optymalizacji należy dążyć do maksymalizacji wskaźnika $\eta$.

Należy też pamiętać, że współczynnik jest poprawnie określony tylko dla wartości $\sigma^2_G > 0$.
Przy czym, wartość $0$ może on przyjąć tylko dla obrazu o jednym poziomie szarości - w takim przypadku trudno mówić o podziale pikseli na dwie klasy (skoro występuje tylko jedna).

Ostatecznie optymalny próg binaryzacji $\bar{k}$ wyliczamy na podstawie zależności:
\begin{equation}
\tag{16}
\sigma^2_B(\bar{k}) \max\limits_{l \in[0,L-1]} {\sigma^2_B(k) }
\end{equation}

Uwagi:
- może się zdarzyć, że znajdziemy więcej niż jedno maksimum tj. więcej wartości $\bar{k}$.
  W takim przypadku zwykle zakłada się, że próg będzie średnią otrzymanych wartości.
- liczby $P_0(\bar{k})$ i $P_1(\bar{k})$ odpowiadają powierzchni zajmowanej przez obiekty klas $C_0$ i $C_1$.
- liczby $m_0(\bar{k})$ i $m_1(\bar{k})$ odpowiadają średniej jasności obiektów klas $C_0$ i $C_1$.
- wartość parametru $\eta(\bar{k})$ określa "jakość" wyznaczonego progu -- im większa tym lepiej.

Zadanie: wykorzystując podane powyżej informacje należy zaimplementować metodę wyznaczania progu binaryzacji zaproponowaną przez Otsu.

1. Wczytaj obraz _coins.png_.
      Wyświetl go.

2. Wyznacz jego histogram znormalizowany oraz oblicz średnią jasność (można do tego wykorzystać histogram) - kod zbliżony do stworzonego wcześniej.

3. Zdefiniuj 256-elementowy wektor na współczynniki $\sigma_B^2$ (funkcja `np.zeros`).

4. W pętli po możliwych wartościach progu binaryzacji wyznacz wartość $\sigma_B^2(k)$ na podstawie zależności $(14)$.
      Uwagi:
      - wcześniejszego liczenia wartości $P_0(k)$ i $m(k)$ można uniknąć inkrementując wartośc $P_0, m$  w każdej iteracji.
      - należy pamiętać, że równanie ma sens tylko dla $0 < P_0(k) < 1$. <br>

5. Wyświetl przebieg $\sigma_B^2(k)$.
      Wykorzystaj funkcję `plt.plot`.

6. Wyznacz wartość $\bar{k}$ dla której współczynnik $\sigma_B^2$ jest maksymalny.
	  Można to zrobić poprzez dodanie instrukcji w pętli (rozwiązanie bardziej elegancie) lub wykorzystując funkcję `max` (rozwiązanie dla leniwych).
	  Uwaga. Proszę pominąć obsługę przypadków niejednoznacznego maksimum.

7. Zbinaryzuj obraz wykorzystując otrzymany próg.
      Porównaj wyniki z rezultatem binaryzacji "ręcznej".

8. W OpenCV dostępna jest implementacja metody Otsu - funkcja `cv2.threshold` z parametrem `cv2.THRESH_OTSU`.
      Funkcja zwraca zbinaryzowany obraz oraz próg.
      Wykonaj binaryzację obrazu _coins.png_ metodą Otsu.
      Porównaj wyniki z własną implementacją - powinno wyjść tak samo (tzn. taki sam próg).

9. Przeprowadź eksperyment również na obrazie _rice.png_ i _catalogue.png_

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def binary_threshold(image, threshold):
    binary = np.zeros_like(image)
    binary[image > threshold] = 255
    return binary

def otsu_threshold(image):
    hist = cv2.calcHist([image], [0], None, [256], [0, 256])
    hist = np.squeeze(hist)
    
    pixel_count = image.shape[0] * image.shape[1]
    p = hist / pixel_count
    
    intensity_levels = np.arange(256)
    mG = np.dot(intensity_levels, p)
    
    sigma_b_squared = np.zeros(256)
    
    P0 = 0
    m = 0
    
    for k in range(256):
        P0 += p[k]
        m += k * p[k]
        
        if P0 > 0 and P0 < 1:
            sigma_b_squared[k] = (mG * P0 - m) ** 2 / (P0 * (1 - P0))
    
    optimal_k = np.argmax(sigma_b_squared)
    
    plt.figure(figsize=(10, 5))
    plt.plot(sigma_b_squared)
    plt.title(f"Przebieg wariancji międzyklasowej σ²ᵦ(k)")
    plt.xlabel("Próg k")
    plt.ylabel("σ²ᵦ(k)")
    plt.grid(True)
    plt.show()
    
    print(f"Optymalny próg metodą Otsu: {optimal_k}")
    
    binary_image = binary_threshold(image, optimal_k)
    
    return optimal_k, binary_image, sigma_b_squared

def otsu_opencv(image):
    ret, binary_opencv = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    plt.figure(figsize=(10, 5))
    plt.imshow(binary_opencv, cmap='gray')
    plt.title(f"Binaryzacja Otsu (OpenCV) - próg={ret}")
    plt.show()
    
    print(f"Próg wyznaczony przez OpenCV: {ret}")
    
    return ret, binary_opencv

print("\nMETODA OTSU DLA OBRAZU COINS.PNG")
print("="*50)
coins_image = cv2.imread("coins.png", cv2.IMREAD_GRAYSCALE)
plt.figure()
plt.imshow(coins_image, cmap='gray')
plt.title("Obraz coins.png")
plt.show()

threshold_otsu, binary_otsu, sigma_b_squared = otsu_threshold(coins_image)

threshold_opencv, binary_opencv = otsu_opencv(coins_image)

print(f"Różnica między progami: {abs(threshold_otsu - threshold_opencv)}")

print("\nMETODA OTSU DLA OBRAZU RICE.PNG")
print("="*50)
rice_image = cv2.imread("rice.png", cv2.IMREAD_GRAYSCALE)
plt.figure()
plt.imshow(rice_image, cmap='gray')
plt.title("Obraz rice.png")
plt.show()

threshold_otsu_rice, binary_otsu_rice, _ = otsu_threshold(rice_image)
threshold_opencv_rice, _ = otsu_opencv(rice_image)

print("\nMETODA OTSU DLA OBRAZU CATALOGUE.PNG")
print("="*50)
catalogue_image = cv2.imread("catalogue.png", cv2.IMREAD_GRAYSCALE)
plt.figure()
plt.imshow(catalogue_image, cmap='gray')
plt.title("Obraz catalogue.png")
plt.show()

threshold_otsu_cat, binary_otsu_cat, _ = otsu_threshold(catalogue_image)
threshold_opencv_cat, _ = otsu_opencv(catalogue_image)

### Binaryzacja lokalna


Analiza wyników binaryzacji dla obrazów _rice.png_ i _catalogue.png_ pokazuje, że globalna binaryzacja nie najlepiej działa dla obrazów o niejednorodnym oświetleniu.
Dla obu obrazów trudno również wyznaczyć odpowiedni próg "ręcznie".

Metodą, która pozwala poprawić wyniki binaryzacji, jest binaryzacja lokalna (niekiedy zwana adaptacyjną).
W jednym z wariantów polega ona na wyznaczeniu progu osobno dla każdego piksela na podstawie jego otoczenia (tj. własności jego kontekstu, okna).

1. Dla uproszczenia zakładamy, że obraz ma rozmiar $256 \times 256$ pikseli. Przyjmijmy okno analizy o rozmiarze 15 pikseli.

2. Najprostsza wersja binaryzacji lokalnej zakłada, że próg binaryzacji dla danego okna to średnia z pikseli w tym oknie.

3. Wczytaj obraz _rice.png_. Rozmiar obrazka (`X,Y`) można uzyskać stosując taką składnię: `(X, Y) = obraz.shape`.

4. Podstawą algorytmu są dwie pętle `for` iterujące po pikselach obrazka:


        for j in range(W/2, Y-W/2):
    
	        for i in range(W/2, X-W/2):


5. Wewnątrz pętli należy dla każdego piksela wyciąć jego otoczenie o rozmiarze `W` (operator `:`), wyznaczyć z niego średnią (metoda `mean`) i na jej podstawie dokonać binaryzacji.

6. Wyświetl obrazy oryginalny i zbinaryzowany.

7. Zaobserwuj rezultaty działania metody dla obrazów _rice.png_ i _catalogue.png_.
      Poeksperymentuj z rozmiarem okna (proszę nie przesadzać z rozmiarem, gdyż istotnie wpływa on na czas obliczeń).
      Jaka jest podstawowa wada zaimplementowanej metody? (pomijając złożoność obliczeniową).
      Proszę się zastanowić co jest źródłem błędów.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def local_threshold(image, window_size=15):
    (X, Y) = image.shape
    binary_image = np.zeros((X, Y), dtype=np.uint8)
    half_window = window_size // 2
    print(f"Wykonywanie binaryzacji lokalnej z rozmiarem okna {window_size}×{window_size}...")
    for j in range(half_window, Y - half_window):
        if j % (Y // 10) == 0:
            print(f"Postęp: {j / Y * 100:.0f}%")
        for i in range(half_window, X - half_window):
            window = image[i - half_window:i + half_window + 1, 
                          j - half_window:j + half_window + 1]
            mean_value = np.mean(window)
            if image[i, j] > mean_value:
                binary_image[i, j] = 255
            else:
                binary_image[i, j] = 0
    
    print("Binaryzacja lokalna zakończona.")
    plt.figure(figsize=(10, 5))
    plt.subplot(121)
    plt.imshow(image, cmap='gray')
    plt.title("Obraz oryginalny")
    
    plt.subplot(122)
    plt.imshow(binary_image, cmap='gray')
    plt.title(f"Binaryzacja lokalna (okno {window_size}×{window_size})")
    plt.show()
    
    return binary_image

print("\nBINARYZACJA LOKALNA DLA OBRAZU RICE.PNG")
print("="*50)
rice_image = cv2.imread("rice.png", cv2.IMREAD_GRAYSCALE)
binary_rice_local = local_threshold(rice_image, window_size=15)

binary_rice_local_small = local_threshold(rice_image, window_size=7)
binary_rice_local_large = local_threshold(rice_image, window_size=31)

print("\nBINARYZACJA LOKALNA DLA OBRAZU CATALOGUE.PNG")
print("="*50)
catalogue_image = cv2.imread("catalogue.png", cv2.IMREAD_GRAYSCALE)
binary_catalogue_local = local_threshold(catalogue_image, window_size=15)

8. Jakość działania binaryzacji lokalnej można poprawić wyznaczając próg za pomocą metody Sauvoli i Pietikainena zaproponowanej w artykule *Adaptive document image binarization*.
Wykorzystuje ona, oprócz średniej, informację o odchyleniu standardowym w danym oknie.
Próg binaryzacji wyznaczany jest na podstawie zależności:
\begin{equation}
\tag{17} 
T = srednia * (1 \pm k * ( \frac{odchStd}{R}-1 ) )
\end{equation}
gdzie: $k$ i $R$ to parametry ($R$ zwykle $128$, a $k$ na początek przyjmij $0.15$), $srednia$ i $odchStd$ to odpowiednio średnia i odchylenie standardowe wyliczone w oknie.

9. Zaimplementuj algorytm Sauvoli - wykorzystaj do wyznaczenia średniej i odchylenia metody `mean()` oraz `std()` liczone dla wycinka (podobnie jak średnia w poprzedniej metodzie).
      
10. Uruchom metodę (uwaga - czas obliczeń nie jest krótki). Przeanalizuj wyniki. Zwróć uwagę, że dodanie informacji o odchyleniu standardowym powinno *poprawić* wyniki binaryzacji.
      Jeżeli dzieje się inaczej, to najprawdopodobniej implementacja zawiera błąd. 
     
11. Zastanów się nad znaczeniem symbolu $\pm$ we wzorze na próg. 
      Kiedy należy zastosować znak $+$, a kiedy $-$.

12. Porównaj jakość binaryzacji lokalnej metodą Sauvoli i z progiem na podstawie średniej. 
      Poeksperymentuj z rozmiarem okna i parametrem k (dla obrazów _rice.png_ i _catalogue.png_).

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from time import time

def local_mean_threshold(image, window_size=15):
    """
    Binaryzacja lokalna oparta na średniej w oknie.
    """
    binary_image = np.zeros_like(image)
    half_window = window_size // 2
    (X, Y) = image.shape
    print(f"Wykonuję binaryzację lokalną (średnia) z oknem {window_size}×{window_size}...")
    start_time = time()
    for j in range(half_window, Y - half_window):
        if j % (Y // 10) == 0:
            print(f"Postęp: {j / (Y-half_window*2) * 100:.0f}%")
        for i in range(half_window, X - half_window):
            window = image[i - half_window:i + half_window + 1, j - half_window:j + half_window + 1]
            mean_value = np.mean(window)
            if image[i, j] > mean_value:
                binary_image[i, j] = 255
            else:
                binary_image[i, j] = 0
    end_time = time()
    print(f"Binaryzacja zakończona w czasie {end_time - start_time:.2f} sekund.")
    return binary_image

def sauvoli_threshold(image, window_size=15, k=0.15, R=128, use_plus=True):
    """
    Binaryzacja lokalna metodą Sauvoli.
    
    Parametry:
    image - obraz wejściowy
    window_size - rozmiar lokalnego okna
    k - parametr metody (domyślnie 0.15)
    R - parametr metody (domyślnie 128)
    use_plus - czy używać znaku + we wzorze (True) czy - (False)
    """
    binary_image = np.zeros_like(image)
    half_window = window_size // 2
    (X, Y) = image.shape
    sign = 1 if use_plus else -1
    print(f"Wykonuję binaryzację lokalną metodą Sauvoli z oknem {window_size}×{window_size}, k={k}, R={R}, znak={'+' if use_plus else '-'}...")
    start_time = time()
    for j in range(half_window, Y - half_window):
        if j % (Y // 10) == 0:
            print(f"Postęp: {j / (Y-half_window*2) * 100:.0f}%")
        for i in range(half_window, X - half_window):
            window = image[i - half_window:i + half_window + 1, j - half_window:j + half_window + 1]
            mean_value = np.mean(window)
            std_value = np.std(window)
            threshold = mean_value * (1 + sign * k * ((std_value / R) - 1))
            if image[i, j] > threshold:
                binary_image[i, j] = 255
            else:
                binary_image[i, j] = 0
    end_time = time()
    print(f"Binaryzacja zakończona w czasie {end_time - start_time:.2f} sekund.")
    return binary_image

def compare_methods(image, image_name, window_sizes=[15], k_values=[0.15]):
    """
    Porównuje różne metody binaryzacji dla podanego obrazu.
    """
    plt.figure(figsize=(10, 8))
    plt.subplot(2, 2, 1)
    plt.imshow(image, cmap='gray')
    plt.title(f"Obraz oryginalny: {image_name}")
    plt.axis('off')
    for i, window_size in enumerate(window_sizes):
        for j, k_value in enumerate(k_values):
            binary_mean = local_mean_threshold(image, window_size=window_size)
            binary_sauvoli_plus = sauvoli_threshold(image, window_size=window_size, k=k_value, use_plus=True)
            binary_sauvoli_minus = sauvoli_threshold(image, window_size=window_size, k=k_value, use_plus=False)
            plt.figure(figsize=(15, 10))
            plt.subplot(2, 2, 1)
            plt.imshow(image, cmap='gray')
            plt.title(f"Obraz oryginalny: {image_name}")
            plt.axis('off')
            plt.subplot(2, 2, 2)
            plt.imshow(binary_mean, cmap='gray')
            plt.title(f"Binaryzacja lokalna (średnia), okno={window_size}")
            plt.axis('off')
            plt.subplot(2, 2, 3)
            plt.imshow(binary_sauvoli_plus, cmap='gray')
            plt.title(f"Sauvoli (znak +), okno={window_size}, k={k_value}")
            plt.axis('off')
            plt.subplot(2, 2, 4)
            plt.imshow(binary_sauvoli_minus, cmap='gray')
            plt.title(f"Sauvoli (znak -), okno={window_size}, k={k_value}")
            plt.axis('off')
            plt.tight_layout()
            plt.show()

rice_image = cv2.imread('rice.png', cv2.IMREAD_GRAYSCALE)
catalogue_image = cv2.imread('catalogue.png', cv2.IMREAD_GRAYSCALE)

print("\nANALIZA OBRAZU RICE.PNG")
print("="*50)
compare_methods(rice_image, "rice.png", window_sizes=[15], k_values=[0.15])
compare_methods(rice_image, "rice.png", window_sizes=[7], k_values=[0.10, 0.20])
compare_methods(rice_image, "rice.png", window_sizes=[31], k_values=[0.15])

print("\nANALIZA OBRAZU CATALOGUE.PNG")
print("="*50)
compare_methods(catalogue_image, "catalogue.png", window_sizes=[15], k_values=[0.15])
compare_methods(catalogue_image, "catalogue.png", window_sizes=[31], k_values=[0.20])

### Binaryzacja dwuprogowa

Binaryzację można przeprowadzić wykorzystując więciej niż jedn próg.
Przykładem jest binaryzacja dwuprogowa - wybieramy w ten sposób przedział jasności (piksele w nim zawarte klasyfikujemy jako obiekty).

1. Wczytaj obraz _bart.png_. 
Wyświetl go, wyznacz i wyświetl jego histogram.
Oceń, który fragment histogramu odpowiada kolorowi skóry Barta Simpsona.
W tym celu wyświetl obraz wykorzystując funkcję cv2.imshow("Tytuł okna", obraz) i wykorzystaj fakt, że przemieszczanie kursora po obrazie wyświetla wartości pikseli.<br>
**UWAGA 1 - W systemie Windows wartości nie wyświetlają się. Aby odczytać wartości pikseli można zapisać obrazek na dysku (`cv2.imwrite('Nazwa.png', Image)`), a następnie odczytać wartościa programem do edycji obrazów, np. *paint*.**<br>
**UWAGA 2 - NIE zamykaj okna z obrazem przez kliknięcie - okno zamknie się po wciśnięciu dowolnego klawisza klawiatury**.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

bart_image = cv2.imread("bart.png", cv2.IMREAD_GRAYSCALE)

plt.figure(figsize=(8, 6))
plt.imshow(bart_image, cmap='gray')
plt.title("Obraz oryginalny - Bart Simpson")
plt.axis('off')
plt.show()

hist_bart = cv2.calcHist([bart_image], [0], None, [256], [0, 256])
hist_bart = np.squeeze(hist_bart)

plt.figure(figsize=(10, 5))
plt.plot(hist_bart)
plt.xlim([0, 256])
plt.xticks(np.arange(0, 256, 20))
plt.title("Histogram obrazu Barta Simpsona")
plt.grid(True)
plt.show()

cv2.imwrite('bart_dla_analizy.png', bart_image)
print("Obraz został zapisany jako 'bart_dla_analizy.png'.")

cv2.imshow("Bart Simpson - analiza kolorów", bart_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

def double_threshold(image, lower_threshold, upper_threshold):
    """
    Binaryzacja dwuprogowa - piksele o wartościach między lower_threshold a upper_threshold
    będą białe (255), pozostałe czarne (0).
    """
    binary_image = np.zeros_like(image)
    binary_image[(image >= lower_threshold) & (image <= upper_threshold)] = 255
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(image, cmap='gray')
    plt.title("Obraz oryginalny")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(binary_image, cmap='gray')
    plt.title(f"Binaryzacja dwuprogowa [{lower_threshold}-{upper_threshold}]")
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return binary_image

lower_threshold = 170
upper_threshold = 190

print("\nBINARYZACJA DWUPROGOWA - WYODRĘBNIANIE KOLORU SKÓRY BARTA")
print("=" * 50)
binary_bart = double_threshold(bart_image, lower_threshold, upper_threshold)

binary_bart_wide = double_threshold(bart_image, 160, 200)
binary_bart_narrow = double_threshold(bart_image, 175, 185)

2. Przeprowadź segmentację na podstawie koloru skóry (binaryzację dwuprogową). 
      Wykorzystaj przekształcenie obrazów z wartościami True, False na wartości 1,0 i mnożenie obrazów.
 
3. Wynik wyświetl.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

bart_image = cv2.imread("bart.png", cv2.IMREAD_GRAYSCALE)

plt.figure(figsize=(8, 6))
plt.imshow(bart_image, cmap='gray')
plt.title("Obraz oryginalny - Bart Simpson")
plt.show()

hist_bart = cv2.calcHist([bart_image], [0], None, [256], [0, 256])
hist_bart = np.squeeze(hist_bart)

plt.figure(figsize=(10, 5))
plt.plot(hist_bart)
plt.xlim([0, 256])
plt.xticks(np.arange(0, 256, 20.0))
plt.title("Histogram obrazu Barta Simpsona")
plt.grid(True)
plt.show()

cv2.imwrite('bart_dla_analizy.png', bart_image)
print("Obraz został zapisany jako 'bart_dla_analizy.png'.")

cv2.imshow("Bart Simpson - analiza kolorów", bart_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

lower_threshold = 170
upper_threshold = 190

skin_mask = (bart_image >= lower_threshold) & (bart_image <= upper_threshold)
skin_mask_binary = skin_mask.astype(np.uint8)
segmented_image = bart_image * skin_mask_binary

plt.figure(figsize=(15, 5))

plt.subplot(131)
plt.imshow(bart_image, cmap='gray')
plt.title("Obraz oryginalny")

plt.subplot(132)
plt.imshow(skin_mask_binary, cmap='gray')
plt.title(f"Maska binarna skóry [{lower_threshold}-{upper_threshold}]")

plt.subplot(133)
plt.imshow(segmented_image, cmap='gray')
plt.title("Wynik segmentacji")

plt.tight_layout()
plt.show()